In [ ]:
# @title 1. Clone LTX-Video repo + install (native inference path)
# The 2B distilled model loads via the repo's own inference API,
# NOT diffusers from_pretrained (that 404s on this repo).
!git clone --depth 1 https://github.com/Lightricks/LTX-Video.git /content/ltx
%cd /content/ltx
!pip install -q -e .
!pip install -q imageio imageio-ffmpeg

In [ ]:
# @title 2. Upload vault_door.jpeg
# Drag vault_door.jpeg into the LEFT sidebar file browser, then run this.
import os
IMAGE_PATH = '/content/vault_door.jpeg'
assert os.path.exists(IMAGE_PATH), 'Upload vault_door.jpeg to the left sidebar first'
print('image present:', IMAGE_PATH)

In [ ]:
# @title 3. Inspect InferenceConfig fields (so we call it correctly, no guessing)
from ltx_video.inference import InferenceConfig
import dataclasses
print('=== InferenceConfig fields ===')
for f in dataclasses.fields(InferenceConfig):
    print(f.name, '|', f.type)

In [ ]:
# @title 4. Run LTX 2B-Distilled image-to-video
# Build the config from the exact fields printed in cell 3.
# Distilled: 8 steps, guidance 1, no STG. Free, fits Colab T4 (15GB).
from ltx_video.inference import infer, InferenceConfig

PROMPT = (
    'Paper-collage animation style, flat illustrated vault door made of cut paper layers, '
    'the paper pieces shift and lift slightly with a gentle hand-made stop-motion drift, '
    'soft paper shadows, warm editorial color palette, Vox-style explainer motion, '
    'subtle slow camera push-in, no 3D, no photorealism'
)
NEG = 'photorealistic, 3d render, CGI, jitter, flicker, distorted, blurry'

cfg = InferenceConfig(
    prompt=PROMPT,
    negative_prompt=NEG,
    media_path=IMAGE_PATH,           # adjust field name if cell 3 showed a different one
    height=1280, width=704,
    num_frames=121,
    seed=42,
    pipeline_config='configs/ltxv-2b-0.9.6-distilled.yaml',
    output_path='/content/ltx_vault.mp4',
)
infer(config=cfg)
print('inference done')


In [ ]:
# @title 5. Download result
import os
from google.colab import files
if os.path.exists('/content/ltx_vault.mp4'):
    files.download('/content/ltx_vault.mp4')
    print('Download triggered. Move to ~/Downloads/vox/ltx_vault.mp4 and ask Hermes to verify >=60%.')
else:
    print('ltx_vault.mp4 not found -- run cell 3 to see the real InferenceConfig field names, then fix cell 4.')